# SCF abstraction: Memory allocation
Since the procedure between SCF and 4c SCF is the same, and we have already defined the non-relativistic SCF routines, the goal is to abstract those to be able to use them with minimal change. Here, we will translate the non-relativistic scf structs and memory allocation routines to be able to use them in the 4c SCF procedure.

In [1]:
from typing import Union, Literal, List
from dataclasses import dataclass

import numpy as np
from numpy.typing import NDArray


from py_mods.src.external.DIRAC_ME import (
    build_S_V_W_T_from_h5,
    get_nuc_charge,
    full_eri_from_h5,
    build_uncontracted_basis_from_h5,
)

from py_mods.src.SCF_4c_dev.KUSCF_dev import (
    occupation_4c,
    eri_classified,
)

# 1. Context class

The context class is similar, but we need some extra matrix elements and information, like the large and small components size and the W matrix.

In [2]:
@dataclass
class CS_4c_KU_SCF_Context:
    """
    Context for complex-scaled four-component Kramers-unrestricted self-consistent field (CS-4c-KU-SCF) calculations.

    Attributes
    ----------
    nL : int
        Number of AO large component basis.
    nS : int
        Number of AO small component basis.
    S : NDArray[np.float64]
        Overlap matrix.
    T : NDArray[np.complex128]
        Kinetic energy matrix.
    V : NDArray[np.float64]
        Nuclear attraction matrix.
    W : NDArray[np.float64]
        V_sC - 2mc^2 matrix.
    eri_classess : NDArray[np.float64]
        Electron repulsion integrals zeroed in non used combinations such as (LS|SS).
    n_electrons : int
        Total electron count (must be an even number).
    theta : float, optional
        Complex-scaling angle in radians. Default is 0.0.
    LC_occupation : int, NDArray[np.int32] or None, optional
        Occupation vector of electronic states.
        If an array, it explicitly defines the occupation numbers.
        If None, a default occupation is built based on `n_electrons`. Default is None.
    max_iter : int, optional
        Maximum number of SCF iterations. Default is 100.
    threshold : float, optional
        Convergence threshold. Default is 1e-12.
    p_guess : {'core', 'ones', 'INPORB'}, optional
        Initial density guess type. Default is 'core'.
    guess_max_iter : int or None, optional
        Maximum iterations for a preliminary RHF guess (if applicable). Default is None.
    initial_orbitals : NDArray[np.float64], NDArray[np.complex128] or None, optional
        Imported orbitals for the initial guess. Required if `p_guess` is 'INPORB'.
        Can be real or complex. Default is None.
    verbose : bool, optional
        If True, print progress information to stdout. Default is False.
    conv_type : {None, 'DIIS', 'CROP'}, optional
        Convergence acceleration algorithm. Default is 'DIIS'.
    acc_hist_size : int, optional
        History size for the convergence acceleration subspace. Default is 10.
    acc_iteration_start : int, optional
        Iteration number to start applying convergence acceleration. Default is 12.
    """

    # Required
    nL : int
    nS : int
    S: NDArray[np.float64]
    T: NDArray[np.complex128]
    V: NDArray[np.float64]
    W : NDArray[np.float64]
    eri_classess: NDArray[np.float64]
    n_electrons: int

    # Optional
    theta: float = 0.0
    LC_occupation: Union[int, NDArray[np.int32], None] = None
    max_iter: int = 100
    threshold: float = 1e-12
    p_guess: Literal["core", "ones", "INPORB"] = "core"
    guess_max_iter: Union[int, None] = None
    initial_orbitals: Union[NDArray[np.float64], NDArray[np.complex128], None] = None
    verbose: bool = False
    conv_type: Literal[None, "DIIS", "CROP"] = "DIIS"
    acc_hist_size: int = 10
    acc_iteration_start: int = 12

    # Internal
    _eigensolver: Literal["eig", "eigh"] = "eigh"
    _convergence_criteria: Literal["norm", "max"] = "norm"

And we have the context validater function:

In [3]:
def validate_CS_4c_KU_SCF_context_input(ctx):
    """
    Validate the input context for CS-4c-KU-SCF calculations.

    Raises
    ------
    ValueError
        If any of the validation checks fail, a ValueError is raised with an appropriate message.
    
    Notes
    -----
    This function checks the following:
    - The dimension of the overlap matrix S (and T and V) must be 2*(nL+nS). 2 * since we have both alpha and beta components with the same AO basis.
    - The dimensions of S, T, and V must be the same.
    - The dimension of T must be 2*dimension of S.
    - The convergence assist type must be either None, 'DIIS', or 'CROP'.
    - The eigensolver must be either 'eig', 'eigh', or 'genh'.
    """
    if not 2 * (ctx.nL + ctx.nS) == len(ctx.S):
        raise ValueError(
            f"Dimension of S (and T, V, W) must be 2*(nL+nS). Got {len(ctx.S)}, expected {2 * (ctx.nL + ctx.nS)}"
        )

    if not len(ctx.S) == len(ctx.W) == len(ctx.V) == len(ctx.T):
        raise ValueError(
            f"Dimension of S, V, W, and T must be the same. Got {len(ctx.S)}, {len(ctx.V)}, {len(ctx.W)},  {len(ctx.T)}"
        )

    if ctx.conv_type not in (None, "DIIS", "CROP"):
        raise ValueError("Convergence assist must be either None, 'DIIS', or 'CROP'")

    if ctx._eigensolver not in [
        "eig",
        "eigh",
        "genh",
    ]:
        raise ValueError(
            f"Eigensolver must be either 'eig', 'eigh' or 'genh'. Got {ctx._eigensolver}"
        )

# 2. SCF constants
We need also to store some other constants like the orthogonalization matrix and the occupation:

In [4]:
@dataclass
class CS_4c_KU_SCF_Constants:
    """
    Constants and scaled/calculated matrices for CS-4c-KU-SCF calculations.

    Attributes
    ----------
    dim : int 
        Total number of basis functions (2*(nL+nS)).
    X : NDArray[np.complex128]
        Orthogonalization matrix (S^{-1/2} for canonical orthogonalization).
    full_det : NDArray[np.int32]
        Occupation vector including small and large components.
    eri_scaled : NDArray[np.complex128]
        Complex-scaled class eris in the atomic orbital basis.
    H_core : NDArray[np.complex128]
        Complex-scaled core Hamiltonian matrix (T + V).
    core_mask : NDArray[np.bool]
        Boolean mask of Hcore interactions.
    _eigensolver : {'eig', 'eigh'}
        Solver to use for diagonalization.
    acc_iteration_start : int, optional
        Iteration number at which to begin convergence acceleration. Default is 10.
    acc_requested : bool, optional
        Boolean indicating if convergence acceleration (DIIS/CROP) is requested. Default is False.
    """

    dim : int 
    X: NDArray[np.complex128]
    full_det: NDArray[np.int32]
    eri_scaled: NDArray[np.complex128]
    H_core: NDArray[np.complex128]
    core_mask: NDArray[np.bool]
    _eigensolver: Literal["eig", "eigh"]
    acc_iteration_start: int = 10
    acc_requested: bool = False

And the allocator function is:

In [5]:
def allocate_CS_4c_KU_SCF_extended_context(
    ctx: CS_4c_KU_SCF_Context,
) -> CS_4c_KU_SCF_Constants:
    """
    Allocate extended context for complex-scaled four-component Kramers-unrestricted self-consistent field (CS-4c-KU-SCF) calculations.

    Parameters
    ----------
    ctx : CS_4c_KU_SCF_Context
        Input context with system parameters.

    Returns
    -------
    CS_4c_KU_SCF_Constants
        Allocated extended context with preallocated matrices.
    """
    nL = ctx.nL
    nS = ctx.nS
    dim = nL + nS
    X = np.zeros((dim, dim), dtype=np.complex128)
    full_det = np.zeros(dim, dtype=np.int32)
    eri_scaled = np.zeros((dim, dim, dim, dim), dtype=np.complex128)
    H_core = np.zeros((dim, dim), dtype=np.complex128)
    core_mask = np.zeros((dim, dim), dtype=np.bool)
    _eigensolver = ctx._eigensolver

    return CS_4c_KU_SCF_Constants(
        dim=dim,
        X=X,
        full_det=full_det,
        eri_scaled=eri_scaled,
        H_core=H_core,
        core_mask=core_mask,
        _eigensolver=_eigensolver,
    )

# 3. SCF state class 
Now the state class, where we include now the positive eigenvalues

In [6]:
@dataclass
class CS_4c_KU_SCF_State:
    """
    State struct for (CS)RHF procedure.

    Attributes
    ----------
    iteration : int
        Current SCF iteration number.
    P : NDArray[np.complex128]
        Current density matrix.
    E_prev : np.complex128
        Energy from previous iteration.
    use_conv_acc : bool
        Bool indicating if convergence acceleration is used.
    F_guess : List[NDArray[np.complex128]]
        DIIS/CROP history of F matrices.
    residuals : List[NDArray[np.complex128]]
        DIIS/CROP history of error vectors.
    F_next : NDArray[np.complex128]
        Extrapolated F matrix.
    error : complex
        Current error.
    converged : bool
        True if converged with threshold.
    C_munu : NDArray[np.complex128]
        MO coefficients in the atomic orbital basis.
    C_prime : NDArray[np.complex128]
        MO coefficients in the orthogonalized basis.
    e_orb : NDArray[np.complex128]
        Current orbital energies.
    e_electronic_orb : NDArray[np.complex128]
        Current electronic solution orbital energies. (size is 2*nL)
    F : NDArray[np.complex128]
        Current Fock matrix.
    r : NDArray[np.complex128]
        Current residual/error vector.
    E_SCF : np.complex128
        Current total RHF energy.
    E_diff : np.complex128
        Change in energy from the previous iteration.
    P_old : NDArray[np.complex128]
        Density matrix from the previous iteration.
    """

    iteration: int
    P: NDArray[np.complex128]
    E_prev: np.complex128
    use_conv_acc: bool
    F_guess: List[NDArray[np.complex128]]
    residuals: List[NDArray[np.complex128]]
    F_next: NDArray[np.complex128]
    error: complex
    converged: bool
    C_munu: NDArray[np.complex128]
    C_prime: NDArray[np.complex128]
    e_orb: NDArray[np.complex128]
    e_electronic_orb : NDArray[np.complex128]
    F: NDArray[np.complex128]
    r: NDArray[np.complex128]
    E_SCF: np.complex128
    E_diff: np.complex128
    P_old: NDArray[np.complex128]

And the allocator:

In [7]:
def allocate_CS_4c_KU_SCF_state(ctx: CS_4c_KU_SCF_Context) -> CS_4c_KU_SCF_State:
    """
    Allocate state for CSRHF calculations.

    Parameters
    ----------
    ctx : CS_4c_KU_SCF_Context
        Input context with system parameters.

    Returns
    -------
    CS_4c_KU_SCF_State
        Allocated state with preallocated matrices and initial values.
    """
    iteration = 0
    P = np.zeros((len(ctx.S), len(ctx.S)), dtype=np.complex128)
    E_prev = np.complex128(0.0)
    use_conv_acc = False
    F_guess: List[NDArray[np.complex128]] = []
    residuals: List[NDArray[np.complex128]] = []
    F_next = np.zeros((len(ctx.S), len(ctx.S)), dtype=np.complex128)
    error: complex = 1e10
    converged: bool = False
    C_munu = np.zeros((len(ctx.S), len(ctx.S)), dtype=np.complex128)
    e_orb = np.zeros(len(ctx.S), dtype=np.complex128)
    e_electronic_orb = np.zeros(ctx.nL * 2, dtype=np.complex128)
    C_prime = np.zeros((len(ctx.S), len(ctx.S)), dtype=np.complex128)
    F = np.zeros((len(ctx.S), len(ctx.S)), dtype=np.complex128)
    r = np.zeros((len(ctx.S), len(ctx.S)), dtype=np.complex128)
    E_SCF = np.complex128(0.0)
    E_diff = np.complex128(0.0)
    P_old = np.zeros((len(ctx.S), len(ctx.S)), dtype=np.complex128)

    return CS_4c_KU_SCF_State(
        iteration=iteration,
        P=P,
        E_prev=E_prev,
        use_conv_acc=use_conv_acc,
        F_guess=F_guess,
        residuals=residuals,
        F_next=F_next,
        error=error,
        converged=converged,
        C_munu=C_munu,
        e_orb=e_orb,
        e_electronic_orb=e_electronic_orb,
        C_prime=C_prime,
        F=F,
        r=r,
        E_SCF=E_SCF,
        E_diff=E_diff,
        P_old=P_old,
    )

# 4. Results and allocator

In [8]:
@dataclass
class CS_4c_KU_SCF_Results:
    """
    Struct containing the results of a (CS)RHF calculation.

    Attributes
    ----------
    context : CSRHFContext
        Input context.
    converged : bool
        True if the SCF converged.
    E_SCF : np.complex128
        Complex-Scaled RHF energy.
    e_orb : NDArray[np.complex128]
        Orbital energies.
    e_electronic_orb : NDArray[np.complex128]
        Electronic solution orbital energies (size is 2*nL).
    n_elec : np.int32
        Final number of electrons.
    full_det : NDArray[np.int32]
        Occupation vector.
    H_core : NDArray[np.complex128]
        (CS)H_core.
    X : NDArray[np.complex128]
        Orthogonalization transformation matrix.
    F_final : NDArray[np.complex128]
        The Fock matrix of the final iteration.
    C_prime : NDArray[np.complex128]
        Molecular orbital coefficients in the orthogonalized basis.
    P_guess : NDArray[np.complex128]
        Initial density matrix.
    P : NDArray[np.complex128]
        Final density matrix.
    C_munu : NDArray[np.complex128]
        Final MO coefficients.
    error : float or complex
        Final error.
    iterations : int
        Number of iterations.
    scaled_eris : NDArray[np.complex128]
        Scaled eris.
    """

    context: CS_4c_KU_SCF_Context
    converged: bool
    E_SCF: np.complex128
    e_orb: NDArray[np.complex128]
    e_electronic_orb: NDArray[np.complex128]
    n_elec: np.int32
    full_det: NDArray[np.int32]
    H_core: NDArray[np.complex128]
    X: NDArray[np.complex128]
    F_final: NDArray[np.complex128]
    C_prime: NDArray[np.complex128]
    P_guess: NDArray[np.complex128]
    P: NDArray[np.complex128]
    C_munu: NDArray[np.complex128]
    error: Union[float, complex]
    iterations: int
    scaled_eris: NDArray[np.complex128]
    homo_index: int

And the allocator

In [9]:
def pack_CS_4c_KU_SCF_results(
    ctx: CS_4c_KU_SCF_Context,
    ext_ctx: CS_4c_KU_SCF_Constants,
    state: CS_4c_KU_SCF_State,
) -> CS_4c_KU_SCF_Results:
    """
    Pack results from (CS)RHF calculation into a results dataclass.

    Parameters
    ----------
    ctx : CS_4c_KU_SCF_Context
        Input context used for the calculation.
    ext_ctx : CS_4c_KU_SCF_Constants
        Extended context with preallocated matrices.
    state : CS_4c_KU_SCF_State
        Final state after SCF iterations.

    Returns
    -------
    CS_4c_KU_SCF_Results
        Packed results from the CS-4c-KU-SCF calculation.
    """
    return CS_4c_KU_SCF_Results(
        context=ctx,
        converged=state.converged,
        E_SCF=state.E_SCF,
        e_orb=state.e_orb,
        e_electronic_orb=state.e_electronic_orb,
        n_elec=np.int32(ctx.n_electrons),
        full_det=ext_ctx.full_det,
        H_core=ext_ctx.H_core,
        X=ext_ctx.X,
        F_final=state.F_next,
        C_prime=state.C_prime,
        P_guess=state.P_old if state.iteration > 0 else state.P,
        P=state.P,
        C_munu=state.C_munu,
        error=state.error,
        iterations=state.iteration,
        scaled_eris=ext_ctx.eri_scaled,
        homo_index=int(np.int32(ctx.n_electrons) / 2),
    )

# Allocation test
We now perform an allocation test with all the needed parameters for the 4c SCF procedure. We will check if the validation matches and if there are no complaints in the allocation. 


In [10]:
h5_filename = "data/He_checkpoint.h5"

S, V, W, T = build_S_V_W_T_from_h5(h5_filename)
H_nuc = V + W + T
nuc_charge = get_nuc_charge(h5_filename)

_, nL, nS = build_uncontracted_basis_from_h5(h5_filename)
eri = full_eri_from_h5(h5_filename)
eri = eri_classified(eri, nL)

occ_det = occupation_4c(nS, nL, 2)

In [11]:
test_ctx = CS_4c_KU_SCF_Context(nL, nS, S, T, V, W, eri, 2)

In [12]:
print(f"Size of S: {S.shape}")
print(f"Size of W: {W.shape}")
print(f"Size of T: {T.shape}")
print(f"Size of V: {V.shape}")
print(f"Size of eri: {eri.shape}")


Size of S: (68, 68)
Size of W: (68, 68)
Size of T: (68, 68)
Size of V: (68, 68)
Size of eri: (34, 34, 34, 34)


In [13]:
repr(test_ctx)

"CS_4c_KU_SCF_Context(nL=9, nS=25, S=array([[1.        , 0.55316115, 0.21345793, ..., 0.        , 0.        ,\n        0.        ],\n       [0.55316115, 1.        , 0.68452017, ..., 0.        , 0.        ,\n        0.        ],\n       [0.21345793, 0.68452017, 1.        , ..., 0.        , 0.        ,\n        0.        ],\n       ...,\n       [0.        , 0.        , 0.        , ..., 3.        , 0.        ,\n        1.        ],\n       [0.        , 0.        , 0.        , ..., 0.        , 1.        ,\n        0.        ],\n       [0.        , 0.        , 0.        , ..., 1.        , 0.        ,\n        3.        ]], shape=(68, 68)), T=array([[ 0.+0.j, -0.-0.j, -0.-0.j, ..., -0.-0.j, -0.-0.j, -0.-0.j],\n       [ 0.+0.j,  0.+0.j, -0.-0.j, ..., -0.-0.j, -0.-0.j, -0.-0.j],\n       [ 0.+0.j,  0.+0.j,  0.+0.j, ..., -0.-0.j, -0.-0.j, -0.-0.j],\n       ...,\n       [-0.+0.j, -0.+0.j, -0.+0.j, ...,  0.-0.j, -0.+0.j, -0.+0.j],\n       [-0.+0.j, -0.+0.j, -0.+0.j, ...,  0.-0.j,  0.-0.j, -0.+0.j]

In [14]:
validate_CS_4c_KU_SCF_context_input(test_ctx)

In [15]:
test_ext_ctx = allocate_CS_4c_KU_SCF_extended_context(test_ctx)
test_state = allocate_CS_4c_KU_SCF_state(test_ctx)
test_results = pack_CS_4c_KU_SCF_results(test_ctx, test_ext_ctx, test_state)